# Assignment: Extend the az.ipynb Lab

**Based on:** `Lab2.ipynb` (the Module 3 lab).

This week's assignment is short on purpose: take your working `Lab2.ipynb` lab and add **one
more step** to the chain. No new concepts, no new setup, no new libraries — just one more
chained LLM call that builds on what you already have.

**Two deployments this time:** `gpt-5.1-ptu` is the default deployment for every existing
step (Steps 1–4). The new step you add (Step 5) must call `gpt-5.4-ptu` instead.

## Step 1 — Start from your working lab

- Make a copy of your completed `Lab2.ipynb` (e.g. rename the copy `assignment3.ipynb`), or
  continue directly inside this notebook — either is fine.
- Copy in your working code from the lab's Steps 1–4: the imports and `.env` config, the
  `AzureOpenAI` client, the `chat()` helper, and the chain itself (fun fact → generate a hard
  question → answer it → evaluate the answer).
- Confirm your `.env`'s `AZURE_APIM_OPENAI_DEPLOYMENT` is set to `gpt-5.1-ptu` — this stays
  the default deployment for Steps 1–4, unchanged.

Run those cells first and confirm they still work before moving on.

## 1. Imports

- `dotenv.load_dotenv` reads key/value pairs from a local `.env` file into environment variables.
- `AzureOpenAI` is the Azure-flavored client class from the `openai` package (as opposed to the
  plain `OpenAI` class used for api.openai.com).

In [ ]:
from dotenv import load_dotenv
import os
import sys

from openai import AzureOpenAI

## 2. Load environment variables

`load_dotenv(override=True)` looks for a `.env` file in the current directory and loads its
contents into `os.environ`. `override=True` means values in `.env` take priority over any values
already set in the shell environment.

Make sure you have a `.env` file (in the same folder as this notebook, or on `sys.path`) that
defines:

```
AZURE_APIM_OPENAI_SUBSCRIPTION_KEY=...
AZURE_APIM_OPENAI_API_VERSION=...
AZURE_APIM_OPENAI_ENDPOINT=https://<your-apim-instance>.azure-api.net
AZURE_APIM_OPENAI_DEPLOYMENT=<your-deployment-name>
```

**Note on the endpoint:** it should be the *host only* — e.g.
`https://apim-azr-ue2-bgpt-prd-ucin.azure-api.net` — **without** a trailing
`/openai/deployments/...` path. The SDK builds the full path itself using the API version and
deployment name.

In [ ]:
# Read .env and override any existing process env values.
load_dotenv(override=True)

# APIM settings from .env. Endpoint must be the host only, e.g.
# https://apim-azr-ue2-bgpt-prd-ucin.azure-api.net  (no /openai/deployments)
api_key = os.getenv("AZURE_APIM_OPENAI_SUBSCRIPTION_KEY")
api_version = os.getenv("AZURE_APIM_OPENAI_API_VERSION")
endpoint = os.getenv("AZURE_APIM_OPENAI_ENDPOINT")
deployment = os.getenv("AZURE_APIM_OPENAI_DEPLOYMENT")

## 3. Validate configuration

Before making any network calls, confirm that all four required settings were actually found.
`all([...])` returns `False` if any of them is `None` or an empty string, in which case
`sys.exit(...)` prints a helpful message and stops execution (raises `SystemExit` — in a notebook
this will show as an error, which is expected if `.env` is missing or incomplete).

If configuration is present, we print a masked preview of the API key (first 8 characters only)
and the deployment name, just to confirm — without ever logging the key can be verified without
leaking the whole secret.

In [ ]:
if not all([api_key, api_version, endpoint, deployment]):
    sys.exit(
        "Missing Azure APIM settings. Set AZURE_APIM_OPENAI_SUBSCRIPTION_KEY, "
        "AZURE_APIM_OPENAI_API_VERSION, AZURE_APIM_OPENAI_ENDPOINT, and "
        "AZURE_APIM_OPENAI_DEPLOYMENT in your .env file."
    )

print(f"Azure APIM key exists and begins {api_key[:8]}")
print(f"Deployment: {deployment}")

## 4. Create the Azure OpenAI client

`AzureOpenAI` is instantiated once and reused for every request. Note the three arguments it
needs, which differ from the plain `OpenAI` client:

- `api_key` — here it's actually the APIM **subscription key**, not an Azure OpenAI resource key,
  since requests are routed through the APIM gateway.
- `api_version` — the Azure OpenAI REST API version (e.g. `2024-06-01`), required because Azure
  versions its API explicitly, unlike api.openai.com.
- `azure_endpoint` — the base host of the APIM instance (see the note above).

In [ ]:
# Sync client pointed at Azure APIM.
client = AzureOpenAI(
    api_key=api_key,
    api_version=api_version,
    azure_endpoint=endpoint,
)

## 5. A small `chat()` helper

This wraps `client.chat.completions.create(...)` so the rest of the notebook doesn't repeat
boilerplate. Two Azure-specific details are worth calling out:

- **`model=deployment`** — On Azure, the `model` parameter is actually the *deployment name* you
  configured in the Azure OpenAI resource (e.g. `gpt-5-chat`), not a generic model id like
  `gpt-5`. Azure routes the request based on that deployment.
- **`max_completion_tokens` instead of `max_tokens`** — Newer reasoning-capable deployments
  (GPT-5-style) reject the older `max_tokens` parameter and require `max_completion_tokens`
  instead. The helper defaults this to `5000` but lets callers override it.

An optional `deployment` argument defaults to the module-level `deployment` from `.env`, so a
later step can override it (e.g. `chat(messages, deployment="gpt-5.4-ptu")`).

In [ ]:
# On Azure the `model` argument is the *deployment name*, not an OpenAI model id.
# GPT-5 deployments need max_completion_tokens (max_tokens is rejected).
def chat(messages, max_completion_tokens=5000, deployment=deployment):
    return client.chat.completions.create(
        model=deployment,
        messages=messages,
        max_completion_tokens=max_completion_tokens,
    )

## 6. Step 1 — Ask for a fun fact

The simplest possible call: a single user message, default token limit, and we print the
model's reply text (`response.choices[0].message.content`).

In [ ]:
# 1) Fun fact
messages = [{"role": "user", "content": "Tell me a short fun fact"}]
response = chat(messages)
fact = response.choices[0].message.content
print(fact)

## 7. Step 2 — Ask the model to invent a hard question

We prompt the model to generate a challenging, IQ-style question, instructing it to respond
**only** with the question itself (no preamble) so that `question` can be reused directly as the
next prompt.

In [ ]:
# 2) Ask the model to invent a hard IQ-style question
question = (
    "Please propose a hard, challenging question to assess someone's IQ. "
    "Respond only with the question."
)
messages = [{"role": "user", "content": question}]
response = chat(messages)
question = response.choices[0].message.content
print(question)

## 8. Step 3 — Ask the model to answer its own question

The `question` text generated in the previous step is now sent back to the model as a fresh
prompt (a brand-new `messages` list — the model has no memory of generating the question; it's
just answering it as if seeing it cold).

Since we're already in a notebook, we can render the answer as nicely formatted Markdown using
`IPython.display` directly — no fallback needed.

In [ ]:
# 3) Ask the model to answer that question
messages = [{"role": "user", "content": question}]
response = chat(messages)
answer = response.choices[0].message.content
print(answer)

In [ ]:
# Render the answer as Markdown in the notebook.
from IPython.display import Markdown, display

display(Markdown(answer))

## 9. Step 4 — Ask the model to evaluate the answer

Finally, we build a single prompt string that includes both the `question` and the `answer`
(using an f-string), and ask the model to judge whether the answer is correct. This is a common
"LLM-as-judge" pattern: use the same (or another) model to self-critique its own prior output.

This call uses a higher `max_completion_tokens` (5000) since an evaluation with reasoning tends
to need more room than a short fun fact. Steps 1–4 all use the default deployment from `.env`.

In [ ]:
# 4) Ask the model to evaluate the answer
message = f"""
Here is a question:
{question}

And here is a possible answer that might be correct or incorrect:
{answer}

Please evaluate if the answer is correct or incorrect.
"""
print(message)

In [ ]:
messages = [{"role": "user", "content": message}]
response = chat(messages, max_completion_tokens=5000)
evaluation = response.choices[0].message.content
print(evaluation)

## Step 2 — Add one more chained step

Add a **5th step** to the chain. Its prompt must be built from at least one variable you
already have (`fact`, `question`, `answer`, or the evaluation text) — the same chaining
pattern as every other step in the lab.

**This step must call `gpt-5.4-ptu`, not the default deployment.** Pass it explicitly when
you call `chat()`:

```python
response = chat(messages, deployment="gpt-5.4-ptu")
```

Pick **one** idea below, or invent your own:

- Rate the difficulty of the question on a 1–10 scale, with a one-sentence justification.
- Rewrite the answer in one simple sentence a 10-year-old could understand.
- Suggest one new, related fun fact that connects to the original topic.
- Translate the final answer into a language of your choice.
- Write a one-line verdict on whether the model's own answer was actually correct, and why.

Store the result in its own variable, and print it clearly labeled (e.g. `=== STEP 5
(gpt-5.4-ptu) ===`).

In [ ]:
# TODO: Step 5 - build a new prompt using an earlier variable
#   (fact, question, answer, and/or the evaluation)
# TODO: call chat(messages, deployment="gpt-5.4-ptu") -- do NOT use the default deployment here
# TODO: extract the result, and print it clearly labeled

## Reflection

Answer in a sentence or two each:

1. **Which earlier variable(s) did your Step 5 prompt use, and why that one?**
2. **What would break if you ran Step 5 before the step it depends on?**
3. **Why might a real project deliberately use a different deployment (e.g. a stronger or
   more expensive model) for just one step in a chain, instead of using it everywhere?**

### My reflection

1.
2.
3.

## Submission checklist

- [ ] Notebook runs top to bottom without errors (`Kernel → Restart & Run All`)
- [ ] `.env` file is **not** included in your submission
- [ ] Step 5 is clearly labeled and its prompt uses at least one earlier variable
- [ ] Reflection questions are answered